# 🎙️ GPT-SoVITS 声音克隆 — 听书club 定制版 v2（Drive 持久化）

> 在 Google Colab 免费 GPU (T4) 上完成**声音克隆 + 语音合成**。中文效果最好的开源方案。
>
> **v2 核心改进**：所有中间文件与模型文件挂载到 **Google Drive**，断网/关闭/重开 **不丢失**。
>
> 适用场景：克隆听书club 主持人声线 → 批量生成读书音频。
>
> ⚠️ 需科学上网访问 Colab。免费 T4 有每日时长限制（约12~24h）。
>
> 流程：`挂载 Drive → 准备音频 → 环境配置(一次) → 启动 WebUI → 零样本克隆 或 完整音频微调训练`

## 第 0 步：准备音频（两种模式）

### 模式A — 快速零样本克隆（10 秒即可，先体验效果）
- 3~10 秒**干净人声**（无 BGM/无杂音/单人/中文普通话）
- 用现有语料截取：`ffmpeg -y -ss 0 -t 15 -i "《某书》.mp3" -acodec pcm_s16le -ar 22050 -ac 1 ref_15s.wav`

### 模式B — 完整音频微调训练（推荐！效果最佳）
- **整本音频（30~50 分钟）直接作为训练语料**，WebUI 会自动切分成数百个 3~10 秒片段并 ASR 标注
- 听书club 现有 120+ 本完整音频都可直接用
- 上传方式：把完整 mp3 放到 **Drive 的 corpus 目录**（`MyDrive/GPT-SoVITS/corpus/`），或直接拖到左侧文件面板后移动到该目录
- 数据越多音色越稳，但训练时间也越长（建议单次用 1~2 本，500~1000 步）

> **注意**：所有语料放 Drive 而非 Colab 本地，否则断连丢失。

In [ ]:
# 检查 GPU（必须为 True，且显存 ≥ 15GB 才适合训练；零样本推理 T4 完全够）
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 第 1 步：挂载 Google Drive（持久化核心，每次会话都要跑）

> 授权一次后，Colab 会记住授权（同一 Google 账号）。
> 所有**预训练模型 / 训练产物 / 切片ASR中间文件 / 训练日志**都存在 Drive，断连重开不丢。

In [ ]:
# 挂载 Google Drive（弹出授权 → 选账号 → 粘贴授权码）
from google.colab import drive
import os

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
    print("✅ Drive 已挂载")
else:
    print("✅ Drive 已挂载（跳过授权）")

# 创建持久化工作区（幂等）
WS = "/content/drive/MyDrive/GPT-SoVITS"
for sub in ["pretrained_models", "SoVITS_weights", "GPT_weights", "output", "logs", "corpus"]:
    os.makedirs(f"{WS}/{sub}", exist_ok=True)

print("持久化目录：", WS)
!ls -la "/content/drive/MyDrive/GPT-SoVITS/" 2>/dev/null || true

## 第 2 步：环境配置（只运行一次，约 15~25 分钟）

> 安装 Anaconda + GPT-SoVITS 代码 + 依赖。
> **预训练模型会直接下载到 Drive**（`MyDrive/GPT-SoVITS/pretrained_models/`），下次硬重置后自动复用、不再下载。
> 之后每次使用：软重连只跑第 1 步 + 第 3 步；硬重置跑 1→2→3（2 会复用 Drive 模型，只重建 conda 环境）。

In [ ]:
# 安装 condacolab（Colab 内的 Anaconda 环境管理）
%pip install -q condacolab
import condacolab
condacolab.install_from_url("https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh")

# 注意：上面会自动重启运行时，重启后请手动回到这里继续执行下一格

In [ ]:
# 初始化：clone 代码 + 建 Drive 软链 + conda 环境 + 安装依赖 + 预训练模型(进 Drive)
%%writefile /content/setup.sh
set -e
WS="/content/drive/MyDrive/GPT-SoVITS"

# 1) 克隆代码（已存在则跳过）
cd /content
if [ ! -d /content/GPT-SoVITS/.git ]; then
  git clone https://github.com/RVC-Boss/GPT-SoVITS.git
fi
cd /content/GPT-SoVITS

# 2) 关键目录 symlink 到 Drive（训练产物/中间文件/预训练模型全持久化）
#    WebUI 读写这些相对路径时自动落到 Drive
mkdir -p "$WS/pretrained_models" "$WS/SoVITS_weights" "$WS/GPT_weights" "$WS/output" "$WS/logs"
for d in SoVITS_weights GPT_weights output logs; do
  if [ ! -L "$d" ]; then
    rm -rf "$d"
    ln -s "$WS/$d" "$d"
    echo "linked: $d -> $WS/$d"
  else
    echo "already linked: $d"
  fi
done
# 预训练模型目录（install.sh 下载到此处即落入 Drive）
if [ ! -L GPT_SoVITS/pretrained_models ]; then
  rm -rf GPT_SoVITS/pretrained_models
  ln -s "$WS/pretrained_models" GPT_SoVITS/pretrained_models
  echo "linked: pretrained_models -> Drive"
fi

# 3) conda 环境（已存在则跳过）
if conda env list | awk '{print $1}' | grep -Fxq "GPTSoVITS"; then
    :
else
    conda create -n GPTSoVITS python=3.10 -y
fi
source activate GPTSoVITS
pip install ipykernel -q

# 4) 安装依赖 + 预训练模型（模型已存在则 install.sh 自动跳过下载）
bash install.sh --device CU126 --source HF --download-uvr5
echo "=== SETUP DONE ==="

!cd /content && bash setup.sh

## 第 3 步：启动 WebUI（每次使用都要运行）

> 启动后输出一个 **gradio.live 公网链接**，点击打开图形界面。

In [ ]:
# 后台启动 WebUI（约 1~3 分钟），自动抓取公网链接
import subprocess, time, re

for p in ["webui.py", "api.py", "api_v2.py"]:
    subprocess.run(["pkill", "-f", p], capture_output=True)

log_path = "/content/webui.log"
with open(log_path, "w") as f:
    proc = subprocess.Popen(
        ["bash", "-lc", "cd /content/GPT-SoVITS && source activate GPTSoVITS && export is_share=True && python webui.py"],
        stdout=f, stderr=subprocess.STDOUT
    )

url = None
for _ in range(180):
    time.sleep(2)
    try:
        log = open(log_path, encoding="utf-8", errors="ignore").read()
    except FileNotFoundError:
        continue
    m = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", log)
    if m:
        url = m.group(0); break
    if "Traceback" in log and "Error" in log:
        print("⚠️ 启动报错，查看最后 30 行日志：")
        print("\n".join(log.strip().splitlines()[-30:]))
        break

if url:
    print("✅ WebUI 已启动！公网链接（请打开）：")
    print("\n" + "="*60)
    print(url)
    print("="*60)
    print("\n若链接打不开，等 10 秒后执行：!tail -50 /content/webui.log 查看新链接")
else:
    print("⏳ 链接尚未出现，查看日志：")
    print("\n".join(open(log_path, encoding="utf-8", errors="ignore").read().strip().splitlines()[-20:]))

## 第 4 步：使用指南

### A. 零样本克隆（最快）
1. WebUI 标签页 **1-GPT-SoVITS-TTS → 1C-推理**
2. 上传参考音频（模式A的 10 秒片段），填写参考音频的**文字内容**（必须与音频一致）
3. 输入要合成的文本 → 「合成语音」→ 试听/下载

### B. 完整音频微调训练（推荐，听书club 主路径）
> 用**整本音频**训练专属模型，音色最像、最稳。中间文件全部自动落 Drive，断连不丢。

1. **语料就位**：确认完整音频在 `MyDrive/GPT-SoVITS/corpus/`（第 1 步创建）
2. 标签页进入 **1-GPT-SoVITS-TTS → 1A-训练集格式化工具**
   - 音频输入目录：填 `/content/drive/MyDrive/GPT-SoVITS/corpus`
   - 输出目录：填 `output/xxx_opt`（默认 `output/slicer_opt`，实际落在 Drive）
   - 勾选「开启自动 ASR 标注」，语言选中文，模型选 fast whisper large-v3
   - 点击「开启处理」→ 自动切分 + ASR 标注，整本 30-50min 音频约 10~20 分钟
   - 处理完成后输出 `.list` 文件路径（记下来）
3. 标签页进入 **1B-微调训练**
   - 实验名填 `tingshu_club`（可多次用不同名，各自独立）
   - 训练数据路径填上一步的输出 `.list`
   - 依次点击「① 解析数据」→「② 训练 Sovits」→「③ 训练 GPT」
   - 训练 500~1000 步即可（T4 约 30~60 分钟），步数越高越像但也易过拟合
4. 回到 **1C-推理**，模型路径选 `tingshu_club_*`，即可用专属音色合成

> 训练中断不怕：`.list`、切片、特征、权重全在 Drive，重开后直接继续/重训。

## 第 5 步（推荐）：一键全自动训练（无需 WebUI 手动操作）

> 上面第 4 步是 WebUI 图形界面方式。如果你更想要**命令行全自动**：
> 放好完整音频到 corpus → 跑下面 5 格 → 自动完成 切分→ASR→格式化→训练 → 模型直接落 Drive。
> 训练完成后用第 6 步 API 或 WebUI 1C 推理。
>
> 🔌 **防断连三件套**（训练前必做）：
> 1. 先跑下面「防断连保活」格（notebook 内 keepalive 线程）
> 2. 浏览器按 F12 → Console → 粘贴防断连 JS（见本步说明格）
> 3. 保持 Colab 标签页在前台、电脑不锁屏不休眠
>
> 即使断了也不怕：每个 Step 产物都在 Drive，重连后从断掉的 Step 重跑即可，已完成步骤自动跳过。

In [ ]:
# 【防断连保活】训练期间保持连接活跃（先跑本格，再跑 Step1-4）
# 原理：每 60s 在前端刷新一次输出，制造"活跃"信号，防止空闲超时断连
import threading, time, IPython.display

KEEPALIVE_STOP = threading.Event()

def keepalive():
    n = 0
    while not KEEPALIVE_STOP.is_set():
        n += 1
        try:
            IPython.display.clear_output(wait=True)
            print(f"🔄 连接保活中… 已持续 {n} 分钟（每 60s 刷新一次）")
        except Exception:
            pass
        time.sleep(60)

th = threading.Thread(target=keepalive, daemon=True)
th.start()
print("✅ 防断连保活已启动（后台线程，每 60s 刷新输出）")
print("提示：保持 Colab 标签页在前台；电脑设置不休眠；最好在浏览器按 F12 再粘贴防断连 JS（见下格说明）")

## 🔌 浏览器级防断连 JS（最有效，推荐配合使用）

> 在浏览器标签页按 **F12** → 切到 **Console** 标签 → 粘贴下面代码 → 回车。
> 脚本每 60s 自动点击 Colab 的 Connect 按钮，从**浏览器层面**阻止空闲断连（比 notebook 内保活更可靠）。

```javascript
// Colab 防断连：粘贴到浏览器 F12 Console 后回车
function ClickConnect(){
  try {
    var btn = document.querySelector('colab-connect-button') ||
              [...document.querySelectorAll('paper-button')].find(function(b){
                return b.textContent && b.textContent.indexOf('Connect') >= 0;
              });
    if (btn) btn.click();
    console.log('keep-alive tick', new Date().toLocaleTimeString());
  } catch(e) { console.log('keep-alive err', e); }
}
setInterval(ClickConnect, 60000);
// 若报错请换用：document.querySelector('colab-connect-button').click()
```

> 效果验证：Console 里每 60s 出现一次 `keep-alive tick` 即生效。
> ⚠️ 若 Colab 界面改版导致选择器失效，日志会打印 keep-alive err，把新版按钮选择器发给我更新脚本。

In [ ]:
# 【自动训练 Step1】切分完整音频为 3-10s 片段（输出落 Drive output/slicer_opt）
import os, subprocess

WS = "/content/drive/MyDrive/GPT-SoVITS"
CORPUS = f"{WS}/corpus"
SLICE_OUT = f"{WS}/output/slicer_opt"
os.makedirs(SLICE_OUT, exist_ok=True)

# 列出语料
corpus_files = [f for f in os.listdir(CORPUS) if f.lower().endswith((".mp3", ".wav", ".m4a", ".flac"))]
print("语料文件：", corpus_files)

if not corpus_files:
    print("❌ corpus 目录为空！请先把完整音频上传到 MyDrive/GPT-SoVITS/corpus/")
else:
    # 幂等：若切片已存在且非空，跳过（断线重跑不浪费）
    existing = [f for f in os.listdir(SLICE_OUT) if f.endswith('.wav')]
    if existing:
        print(f"⏭️ 检测到已有 {len(existing)} 个切片，跳过切分（如想重新切分请清空 {SLICE_OUT}）")
    else:
        # slice_audio.py 参数: inp opt_root threshold min_length min_interval hop_size max_sil_kept _max alpha i_part all_part
        cmd = ["bash", "-lc",
               f"cd /content/GPT-SoVITS && source activate GPTSoVITS && "
               f"python tools/slice_audio.py {CORPUS} {SLICE_OUT} -40 3000 500 10 400 0.9 0.25 0 1 2>&1 | tail -5"]
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
        print(r.stdout[-800:] if r.stdout else "")
        if r.returncode != 0:
            print("STDERR:", r.stderr[-500:])
    n = len([f for f in os.listdir(SLICE_OUT) if f.endswith('.wav')])
    print(f"✅ 切片就绪：{n} 个（Drive output/slicer_opt）")

In [ ]:
# 【自动训练 Step2】ASR 自动标注（Whisper 转写每段文本，输出 .list 到 Drive）
import os, subprocess

WS = "/content/drive/MyDrive/GPT-SoVITS"
SLICE_OUT = f"{WS}/output/slicer_opt"
ASR_OUT = f"{WS}/output/asr_opt"
os.makedirs(ASR_OUT, exist_ok=True)

cmd = ["bash", "-lc",
       f"cd /content/GPT-SoVITS && source activate GPTSoVITS && "
       f"python tools/asr/fasterwhisper_asr.py -i {SLICE_OUT} -o {ASR_OUT} -s large-v3 -l zh -p int8 2>&1 | tail -10"]
r = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
print(r.stdout[-800:] if r.stdout else "")
if r.returncode != 0:
    print("STDERR:", r.stderr[-500:])

# 找到生成的 .list 文件
lists = [f for f in os.listdir(ASR_OUT) if f.endswith(".list")]
# 幂等：若 .list 已存在且非空，跳过（断线重跑不浪费）
if lists:
    list_path = os.path.join(ASR_OUT, lists[0])
    with open(list_path, encoding="utf-8", errors="ignore") as f:
        cnt = len(f.read().strip().split("\n")) if f.read() else 0
    print(f"⏭️ 检测到已有标注文件 {lists[0]}，跳过 ASR")
else:
    cmd = ["bash", "-lc",
           f"cd /content/GPT-SoVITS && source activate GPTSoVITS && "
           f"python tools/asr/fasterwhisper_asr.py -i {SLICE_OUT} -o {ASR_OUT} -s large-v3 -l zh -p int8 2>&1 | tail -10"]
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
    print(r.stdout[-800:] if r.stdout else "")
    if r.returncode != 0:
        print("STDERR:", r.stderr[-500:])
    lists = [f for f in os.listdir(ASR_OUT) if f.endswith(".list")]
    list_path = os.path.join(ASR_OUT, lists[0]) if lists else None

print("生成标注文件：", lists)
if list_path:
    # 显示前几行供校对
    with open(list_path, encoding="utf-8", errors="ignore") as f:
        lines = f.read().strip().split("\n")[:3]
    print("标注样例（前3条）：")
    for ln in lines: print(" ", ln[:120])
    # 保存路径供下一步使用
    with open("/content/asr_list_path.txt", "w") as f:
        f.write(list_path)
    print("\n✅ ASR 就绪 →", list_path)

In [ ]:
# 【自动训练 Step3】格式化数据集 + 训练（SoVITS → GPT，产出落 Drive）
# 耗时：格式化 5~10min；训练 500 步约 30~60min（T4）
import os, subprocess, time

WS = "/content/drive/MyDrive/GPT-SoVITS"
EXP_NAME = "tingshu_club"

try:
    list_path = open("/content/asr_list_path.txt").read().strip()
except Exception:
    list_path = os.path.join(WS, "output/asr_opt", "slicer_opt.list")

EXP_ROOT = "/content/GPT-SoVITS/logs"   # 软链到 Drive 的 logs
os.makedirs(f"{EXP_ROOT}/{EXP_NAME}", exist_ok=True)

def run_train(cmd, timeout=3600):
    r = subprocess.run(["bash", "-lc", f"cd /content/GPT-SoVITS && source activate GPTSoVITS && {cmd}"],
                       capture_output=True, text=True, timeout=timeout)
    tail = (r.stdout or "")[-600:]
    print(tail)
    if r.returncode != 0:
        print("STDERR:", (r.stderr or "")[-500:])
    return r.returncode

# 幂等：若特征文件已生成，跳过整个格式化（断线重跑不浪费）
if os.path.exists(f"{EXP_ROOT}/{EXP_NAME}/6-name2semantic.tsv"):
    print("⏭️ 检测到 6-name2semantic.tsv 已存在，跳过格式化（如想重来请删除 logs/tingshu_club）")
    print("✅ 数据集已就绪（Drive logs/tingshu_club/）")
else:
    # 1) 文本与特征提取（prepare_datasets 1/2/3 步，env 传参）
    print("=== 1) 文本分词与特征提取 ===")
    env = {
        "inp_text": list_path,
        "inp_wav_dir": f"{WS}/output/slicer_opt",
        "exp_name": EXP_NAME,
        "opt_dir": f"{EXP_ROOT}/{EXP_NAME}",
        "bert_pretrained_dir": "/content/GPT-SoVITS/GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large",
        "i_part": "0", "all_parts": "1", "is_half": "True",
    }
    cmd = "env " + " ".join(f'{k}="{v}"' for k, v in env.items()) + " python -s GPT_SoVITS/prepare_datasets/1-get-text.py"
    rc1 = run_train(cmd, timeout=1800)
    if rc1 != 0: print("⚠️ 1-get-text 失败"); raise SystemExit

    print("=== 2) HuBERT 特征 ===")
    cmd2 = "env " + " ".join(f'{k}="{v}"' for k, v in env.items()) + " python -s GPT_SoVITS/prepare_datasets/2-get-hubert-wav32k.py"
    rc2 = run_train(cmd2, timeout=1800)
    if rc2 != 0: print("⚠️ 2-get-hubert 失败"); raise SystemExit

    print("=== 3) 语义 token ===")
    cmd3 = "env " + " ".join(f'{k}="{v}"' for k, v in env.items()) + " python -s GPT_SoVITS/prepare_datasets/3-get-semantic.py"
    rc3 = run_train(cmd3, timeout=1800)
    if rc3 != 0: print("⚠️ 3-get-semantic 失败"); raise SystemExit

    print("\n✅ 数据集格式化完成，产物在 Drive logs/tingshu_club/")

In [ ]:
# 【自动训练 Step4】正式训练（SoVITS 声学模型 → GPT 语义模型）
# 训练时长：500 步约 30~60min。产出 .ckpt/.pth 自动存 Drive GPT_weights / SoVITS_weights
import os, subprocess, json

WS = "/content/drive/MyDrive/GPT-SoVITS"
EXP_NAME = "tingshu_club"
EXP_ROOT = "/content/GPT-SoVITS/logs"
TOTAL_EPOCH = 10        # 或改小：5（更快，音色稍差）；调大：15（更像，更慢）
BATCH_SIZE = 4          # T4 16GB 建议 4；OOM 改 2

# 幂等：模型权重已存在则跳过对应训练（断线重跑不浪费）
sovits_done = os.path.exists(f"{WS}/SoVITS_weights") and any(f.endswith('.pth') for f in os.listdir(f"{WS}/SoVITS_weights"))
gpt_done = os.path.exists(f"{WS}/GPT_weights") and any(f.endswith('.ckpt') for f in os.listdir(f"{WS}/GPT_weights"))

# SoVITS (s2) 训练
if sovits_done:
    print("⏭️ SoVITS 权重已存在，跳过 s2 训练")
else:
    print("=== 训练 SoVITS 声学模型 ===")
    cmd = ["bash", "-lc",
           f"cd /content/GPT-SoVITS && source activate GPTSoVITS && "
           f"python GPT_SoVITS/s2_train.py --config_file GPT_SoVITS/configs/s2.json "
           f"--train_semantic_path {EXP_ROOT}/{EXP_NAME}/6-name2semantic.tsv "
           f"--train_phoneme_path {EXP_ROOT}/{EXP_NAME}/2-name2text.txt "
           f"--output_dir {EXP_ROOT}/{EXP_NAME}/logs_s2_v4 --epochs {TOTAL_EPOCH} --batch_size {BATCH_SIZE} 2>&1 | tail -15"]
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=5400)
    print((r.stdout or "")[-800:])
    if r.returncode != 0:
        print("STDERR:", (r.stderr or "")[-500:])

# GPT (s1) 训练
if gpt_done:
    print("⏭️ GPT 权重已存在，跳过 s1 训练")
else:
    print("\n=== 训练 GPT 语义模型 ===")
    cmd2 = ["bash", "-lc",
            f"cd /content/GPT-SoVITS && source activate GPTSoVITS && "
            f"python GPT_SoVITS/s1_train.py --config_file GPT_SoVITS/configs/s1longer.yaml "
            f"--train_semantic_path {EXP_ROOT}/{EXP_NAME}/6-name2semantic.tsv "
            f"--train_phoneme_path {EXP_ROOT}/{EXP_NAME}/2-name2text.txt "
            f"--output_dir {EXP_ROOT}/{EXP_NAME}/logs_s1_v4 --epochs {TOTAL_EPOCH} --batch_size {BATCH_SIZE} 2>&1 | tail -15"]
    r2 = subprocess.run(cmd2, capture_output=True, text=True, timeout=5400)
    print((r2.stdout or "")[-800:])
    if r2.returncode != 0:
        print("STDERR:", (r2.stderr or "")[-500:])

print("\n✅ 训练完成！模型在 Drive：")
print("  GPT_weights/  →", os.listdir(f"{WS}/GPT_weights") if os.path.exists(f"{WS}/GPT_weights") else "（检查日志）")
print("  SoVITS_weights/ →", os.listdir(f"{WS}/SoVITS_weights") if os.path.exists(f"{WS}/SoVITS_weights") else "（检查日志）")

## 第 6 步（可选）：API 批量合成 + 长文本分段

启动 OpenAI 兼容 API（端口 9880），配合脚本实现「长文本分段 → 批量合成 → ffmpeg 合并」，接入听书club 生产流水线。训练好的模型也存 Drive，API 直接调用。

In [ ]:
# 启动 API 服务（v2，OpenAI 兼容，端口 9880）
import subprocess, time, socket

api_log = "/content/api.log"
with open(api_log, "w") as f:
    proc = subprocess.Popen(
        ["bash", "-lc",
         "cd /content/GPT-SoVITS && source activate GPTSoVITS && "
         "python api_v2.py -a 127.0.0.1 -p 9880 -c GPT_SoVITS/configs/tts_infer.yaml 2>&1"],
        stdout=f, stderr=subprocess.STDOUT
    )

ready = False
for _ in range(120):
    time.sleep(2)
    s = socket.socket(); s.settimeout(1)
    if s.connect_ex(("127.0.0.1", 9880)) == 0:
        ready = True; s.close(); break
    s.close()

if ready:
    print("✅ API 服务已就绪 (127.0.0.1:9880)，支持 OpenAI /v1/audio/speech 接口")
else:
    print("⚠️ API 未就绪，日志：")
    print(open(api_log, encoding="utf-8", errors="ignore").read()[-1000:])

In [ ]:
# API 调用示例（零样本克隆：传参考音频 + 文本 → 返回合成音频）
import requests, base64

# 1) 上传参考音频获得 reference_id（只需做一次）
#    参考音频建议放在 Drive，断连不丢：/content/drive/MyDrive/GPT-SoVITS/corpus/ref_15s.wav
ref_audio = "/content/drive/MyDrive/GPT-SoVITS/corpus/ref_15s.wav"
ref_text  = "大家好，歡迎來到《听书club》，今天為你解讀的是"  # 改成参考音频的实际文字

with open(ref_audio, "rb") as f:
    base64_audio = base64.b64encode(f.read()).decode()

resp = requests.post("http://127.0.0.1:9880/change_refer", json={
    "refer_wav_path": ref_audio,
    "prompt_text": ref_text,
    "prompt_language": "zh"
}, timeout=60)
print("change_refer:", resp.json())

# 2) 合成文本（OpenAI 兼容格式）
text = "大家好，歡迎來到《听书club》。今天為你解讀的是《反脆弱》：那些杀不死我们的，终将使我们更强大。"
r = requests.post("http://127.0.0.1:9880/v1/audio/speech", json={
    "model": "GPT-SoVITS",
    "input": text,
    "voice": "default"
}, timeout=120)
print("HTTP:", r.status_code, "bytes:", len(r.content))
if r.status_code == 200:
    with open("/content/output_api.wav", "wb") as f:
        f.write(r.content)
    print("✅ 已保存 /content/output_api.wav （可在左侧文件面板下载，或拷回 Drive 保存）")

## 长文本批量合成（听书club 10,000 字脚本）

```python
import re, subprocess, requests

text = open("/content/drive/MyDrive/GPT-SoVITS/corpus/script.txt", encoding="utf-8").read()
# 按句切分，每段 ≤ 300 字（保证合成稳定）
chunks, cur = [], ""
for s in re.split(r"(?<=[。！？\n])", text):
    if len(cur) + len(s) > 300: chunks.append(cur); cur = s
    else: cur += s
if cur: chunks.append(cur)

parts = []
for i, ch in enumerate(chunks):
    r = requests.post("http://127.0.0.1:9880/v1/audio/speech",
                      json={"model": "GPT-SoVITS", "input": ch, "voice": "default"}, timeout=180)
    if r.status_code == 200:
        p = f"/content/part_{i:03d}.wav"; open(p, "wb").write(r.content); parts.append(p)
        print(i, "OK", len(ch), "字")

# 合并（输出到 Drive，断连不丢）
with open("/content/concat.txt", "w") as f:
    for p in parts: f.write(f"file '{p}'\n")
subprocess.run(["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", "/content/concat.txt",
                "-c", "copy", "/content/drive/MyDrive/GPT-SoVITS/corpus/final_clone.wav"])
print("✅ 完成：" + str(len(parts)) + " 段，已存 Drive corpus/final_clone.wav")
```

## ⚠️ 断线恢复 & 常见问题

### 断线/重开恢复流程（核心！）
| 情况 | 处理 |
|------|------|
| **软重连**（Colab 顶部重连，磁盘保留） | 跑第 1 步（挂 Drive）→ 跑第 3 步（启动 WebUI），环境还在，秒级恢复 |
| **硬重置**（关闭太久/换机器，/content 清空） | 跑 1→2→4：重装环境约 15-25min，但**模型/中间文件/语料全在 Drive**，install.sh 检测到预训练模型已存在会跳过下载，直接继续训练/推理 |
| **训练中断** | 切片/ASR/.list/特征/权重都在 Drive，重新打开后从断点继续 |
| **第5步自动训练中断** | 每个 Step 有幂等检查：重连后从断掉的 Step 重跑即可，已完成步骤自动跳过（切片存在跳过Step1、.list存在跳过Step2、6-name2semantic.tsv存在跳过Step3、权重存在跳过Step4） |

### 其他
- **参考音频文字必须准确**：零样本克隆相似度 80% 取决于参考音频与文字一致性，用 Whisper 转写后校对
- **模型位置**：训练产物在 `MyDrive/GPT-SoVITS/GPT_weights/` 和 `SoVITS_weights/`，永远在 Drive，无需手动备份
- **Drive 空间**：免费 15GB。一份训练约 1-2GB，预训练模型约 2GB；装不下时在 Drive 清理旧实验
- **速度**：训练时中间文件在 Drive，比本地盘略慢（可接受）；若明显拖慢，把 `logs/` 软链改回本地 `rm logs && mkdir logs`，训练完拷回 Drive
- **声音不像**：换更干净的参考音频、增加微调步数、参考音频用同性别同语速
- **显存不足**：T4 16GB 训练 1000 步没问题；报 OOM 就换新实验名重训（历史权重占显存）

---
*Notebook 由 Hermes Agent 生成 · v2 Drive 持久化 · 基于 RVC-Boss/GPT-SoVITS 官方 Colab-WebUI.ipynb*